# Healthcare Data Analytics — Python EDA

**Portfolio project component:** Exploratory Data Analysis using Python, Pandas, NumPy, Matplotlib and Seaborn.

This notebook uses the same healthcare dataset used for the project's SQL analysis and Power BI dashboard. The objective is to assess data quality, prepare analytical fields, explore patient and admission patterns, analyze billing and length of stay, and summarize business-relevant findings.

> **Data note:** The raw dataset contains 55,500 records and 534 exact duplicate rows. The notebook reports results on the raw dataset for reconciliation with the existing Power BI model, and uses a deduplicated analytical dataframe for analysis where appropriate.


## 1. Import libraries and set display options

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")


## 2. Load the healthcare dataset

The notebook expects the dataset in the project's `data` folder:

`../dataset/healthcare_dataset.csv`

If the notebook is being run outside the GitHub repository structure, change `DATA_PATH` to the local CSV location.


In [ ]:
from pathlib import Path

DATA_PATH = Path("../dataset/healthcare_dataset.csv")

if not DATA_PATH.exists():
    # Fallback for local execution from this notebook's generated workspace
    DATA_PATH = Path("/mnt/dataset/healthcare_dataset.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


## 3. Initial data inspection

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
print(f"Records: {len(df):,}")
print(f"Columns: {df.shape[1]}")


## 4. Data quality assessment

We check for missing values, exact duplicate rows and inconsistent data types before analysis.


In [ ]:
quality = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
}).sort_values("Missing Values", ascending=False)

display(quality)

duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows: {duplicate_count:,}")


### Duplicate-record assessment

There are 534 exact duplicate rows in the raw dataset. These are complete row-for-row duplicates rather than merely repeated patient names. For analytical summaries where each row should represent one unique record, we create a deduplicated dataframe while retaining the raw dataframe for reconciliation.


In [ ]:
df_clean = df.drop_duplicates().copy()

print(f"Raw records:        {len(df):,}")
print(f"Duplicate rows:     {df.duplicated().sum():,}")
print(f"Deduplicated rows:  {len(df_clean):,}")


## 5. Data preparation

Admission and discharge dates are converted to datetime. A derived **Length of Stay** field is created in days.


In [ ]:
date_columns = ["Date of Admission", "Discharge Date"]

for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

df_clean["Length of Stay"] = (
    df_clean["Discharge Date"] - df_clean["Date of Admission"]
).dt.days

print("Date ranges:")
print(f"Admission: {df_clean['Date of Admission'].min().date()} to {df_clean['Date of Admission'].max().date()}")
print(f"Discharge: {df_clean['Discharge Date'].min().date()} to {df_clean['Discharge Date'].max().date()}")

print("\nLength of stay summary:")
display(df_clean["Length of Stay"].describe().to_frame("Days"))


## 6. Descriptive statistics

In [ ]:
numeric_summary = df_clean[[
    "Age", "Billing Amount", "Room Number", "Length of Stay"
]].describe().T

display(numeric_summary)


## 7. Patient profile

We examine gender, age and blood-group distributions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

gender_counts = df_clean["Gender"].value_counts()
gender_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Patient Count by Gender")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Patients")
axes[0].tick_params(axis="x", rotation=0)

age_bins = [0, 18, 30, 45, 60, 75, 100]
age_labels = ["0–17", "18–29", "30–44", "45–59", "60–74", "75+"]
age_group = pd.cut(df_clean["Age"], bins=age_bins, labels=age_labels, right=False)
age_group.value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Patient Count by Age Group")
axes[1].set_xlabel("Age Group")
axes[1].set_ylabel("Patients")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


## 8. Medical condition analysis

In [ ]:
condition_counts = (
    df_clean["Medical Condition"]
    .value_counts()
    .sort_values(ascending=True)
)

plt.figure(figsize=(9, 5))
condition_counts.plot(kind="barh")
plt.title("Patient Volume by Medical Condition")
plt.xlabel("Patients")
plt.ylabel("Medical Condition")
plt.tight_layout()
plt.show()


## 9. Admission and insurance analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

admission_counts = df_clean["Admission Type"].value_counts()
admission_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Patients by Admission Type")
axes[0].set_xlabel("Admission Type")
axes[0].set_ylabel("Patients")
axes[0].tick_params(axis="x", rotation=0)

insurance_counts = df_clean["Insurance Provider"].value_counts().sort_values(ascending=True)
insurance_counts.plot(kind="barh", ax=axes[1])
axes[1].set_title("Patients by Insurance Provider")
axes[1].set_xlabel("Patients")
axes[1].set_ylabel("Insurance Provider")

plt.tight_layout()
plt.show()


## 10. Billing analysis

In [ ]:
billing_by_condition = (
    df_clean.groupby("Medical Condition")["Billing Amount"]
    .agg(["count", "sum", "mean"])
    .sort_values("mean", ascending=False)
)

display(billing_by_condition)

plt.figure(figsize=(9, 5))
billing_by_condition["mean"].sort_values().plot(kind="barh")
plt.title("Average Billing Amount by Medical Condition")
plt.xlabel("Average Billing Amount")
plt.ylabel("Medical Condition")
plt.tight_layout()
plt.show()


In [ ]:
billing_by_insurance = (
    df_clean.groupby("Insurance Provider")["Billing Amount"]
    .agg(["count", "sum", "mean"])
    .sort_values("mean", ascending=False)
)

display(billing_by_insurance)

plt.figure(figsize=(9, 5))
billing_by_insurance["mean"].sort_values().plot(kind="barh")
plt.title("Average Billing Amount by Insurance Provider")
plt.xlabel("Average Billing Amount")
plt.ylabel("Insurance Provider")
plt.tight_layout()
plt.show()


## 11. Length-of-stay analysis

In [ ]:
stay_by_condition = (
    df_clean.groupby("Medical Condition")["Length of Stay"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

display(stay_by_condition)

plt.figure(figsize=(9, 5))
stay_by_condition["mean"].sort_values().plot(kind="barh")
plt.title("Average Length of Stay by Medical Condition")
plt.xlabel("Average Length of Stay (days)")
plt.ylabel("Medical Condition")
plt.tight_layout()
plt.show()


## 12. Billing versus length of stay

A scatter plot is used to examine whether longer stays are associated with higher billing amounts. The chart is exploratory and does not by itself establish causation.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_clean,
    x="Length of Stay",
    y="Billing Amount",
    alpha=0.35
)
plt.title("Billing Amount vs Length of Stay")
plt.xlabel("Length of Stay (days)")
plt.ylabel("Billing Amount")
plt.tight_layout()
plt.show()


## 13. Test results and medication analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

test_counts = df_clean["Test Results"].value_counts()
test_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Patient Count by Test Result")
axes[0].set_xlabel("Test Result")
axes[0].set_ylabel("Patients")
axes[0].tick_params(axis="x", rotation=0)

med_counts = df_clean["Medication"].value_counts().sort_values(ascending=True)
med_counts.plot(kind="barh", ax=axes[1])
axes[1].set_title("Medication Frequency")
axes[1].set_xlabel("Patients")
axes[1].set_ylabel("Medication")

plt.tight_layout()
plt.show()


## 14. Consolidated healthcare KPI summary

In [ ]:
kpis = pd.Series({
    "Analytical Patients": len(df_clean),
    "Total Billing": df_clean["Billing Amount"].sum(),
    "Average Billing": df_clean["Billing Amount"].mean(),
    "Average Length of Stay (days)": df_clean["Length of Stay"].mean(),
    "Median Patient Age": df_clean["Age"].median()
})

display(kpis.to_frame("Value"))


## 15. Key business insights

The following observations are calculated directly from the dataset used in this notebook.

- The raw dataset contains **55,500 records**, with **534 exact duplicate rows**. The deduplicated analytical dataset therefore contains **54,966 records**.
- There are **no missing values** across the dataset columns.
- **Arthritis** has the largest patient volume, with **9,308 records** in the deduplicated analytical dataset.
- **Obesity** has the highest average billing amount among the medical conditions, at approximately **₹25,806**.
- **Asthma** has the longest average length of stay, at approximately **15.7 days**.
- The overall average billing amount is approximately **₹25,539**, while total billing across the raw records is approximately **₹1,417,432,043**.
- The average length of stay is approximately **15.5 days**.

These findings should be interpreted as descriptive analysis of this dataset rather than clinical conclusions.


## 16. Conclusion

This Python analysis complements the project's SQL and Power BI work:

- **SQL** validates the data and answers structured business questions.
- **Python** performs data preparation, exploratory analysis and statistical visualization.
- **Power BI** presents the findings in an interactive dashboard.

Together, these components demonstrate an end-to-end data analytics workflow from raw data quality assessment through business reporting.
